In [ ]:
yeonu@gpusystem:/nas/arpa_h_repository/public_data/CRC_Atlas_1/CRC-Atlas-Split/crc-atlas/Deg_data/DEG_H5AD_Original/result$ ls
B_cell_Wilcoxon_DEG_crcatlas.csv           Mini_10percent_Wilcoxon_DEG_crcatlas.csv  Schwann_cell_Wilcoxon_DEG_crcatlas.csv
Cancer_cell_Wilcoxon_DEG_crcatlas.csv      Myeloid_cell_Wilcoxon_DEG_crcatlas.csv    Stromal_cell_Wilcoxon_DEG_crcatlas.csv
Epithelial_cell_Wilcoxon_DEG_crcatlas.csv  Neutrophil_Wilcoxon_DEG_crcatlas.csv      T_cell_Wilcoxon_DEG_crcatlas.csv
ILC_Wilcoxon_DEG_crcatlas.csv              NK_Wilcoxon_DEG_crcatlas.csv
Mast_cell_Wilcoxon_DEG_crcatlas.csv        Plasma_cell_Wilcoxon_DEG_crcatlas.csv

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. DEG_Wilcoxon 전처리 
# Wilcoxon_DEG : Wilcoxon_DEG_~.csv

df_wilcoxon = pd.read_csv('/nas/arpa_h_repository/public_data/CRC_Atlas_1/CRC-Atlas-Split/crc-atlas/Deg_data/DEG_H5AD_Original/result/Schwann_cell_Wilcoxon_DEG_crcatlas.csv')
print(df_wilcoxon.shape)
print(df_wilcoxon.head())
df_wilcoxon = df_wilcoxon[["names", "logfoldchanges", "pvals", "pvals_adj"]]
df_wilcoxon.rename(columns={"names": "gene","pvals": "pval","pvals_adj": "Wilcoxon_adj_pval"}, inplace=True)
print(df_wilcoxon.head())

(28476, 5)
     names     scores  logfoldchanges          pvals      pvals_adj
0   IGFBP7  34.756287        1.407640  1.113706e-264  3.171388e-260
1  MT-ND4L  27.423323        2.511231  1.445840e-165  1.372391e-161
2    ITM2B  25.698805        0.923202  1.205370e-145  6.864824e-142
3    PNRC1  25.302174        1.250677  3.023014e-141  1.434723e-137
4  MT-ATP8  23.844517        2.617006  1.154379e-125  4.696013e-122
      gene  logfoldchanges           pval  Wilcoxon_adj_pval
0   IGFBP7        1.407640  1.113706e-264      3.171388e-260
1  MT-ND4L        2.511231  1.445840e-165      1.372391e-161
2    ITM2B        0.923202  1.205370e-145      6.864824e-142
3    PNRC1        1.250677  3.023014e-141      1.434723e-137
4  MT-ATP8        2.617006  1.154379e-125      4.696013e-122


In [3]:
df_wilcoxon.head(2)

,gene,logfoldchanges,pval,Wilcoxon_adj_pval
0,IGFBP7,1.407640,1.113706e-264,3.171388e-260
1,MT-ND4L,2.511231,1.445840e-165,1.372391e-161


## <font color="red">**데이터프레임**

In [4]:
import pandas as pd
from functools import reduce

# 1) 미리 각 df에서 gene + adj_pval 컬럼만 추출
dfs = [
    df_wilcoxon[["gene", "logfoldchanges", "pval", "Wilcoxon_adj_pval"]],
    # df_limmatrend[["gene", "limmatrend_adj_pval"]],
    # df_RISCQP[["gene", "RISCQP_adj_pval"]],
    # df_MAST[["gene", "MAST_adj_pval"]],
    # df_monocle_filtered[["gene", "monocle_adj_pval"]],
]

# 2) Reduce를 이용해 순차적으로 outer merge
df_mmm = reduce(
    lambda left, right: pd.merge(left, right, on="gene", how="outer"),
    dfs
)

# 결과 확인
print(df_mmm.shape)
print(df_mmm.head())

(28476, 4)
      gene  logfoldchanges           pval  Wilcoxon_adj_pval
0   IGFBP7        1.407640  1.113706e-264      3.171388e-260
1  MT-ND4L        2.511231  1.445840e-165      1.372391e-161
2    ITM2B        0.923202  1.205370e-145      6.864824e-142
3    PNRC1        1.250677  3.023014e-141      1.434723e-137
4  MT-ATP8        2.617006  1.154379e-125      4.696013e-122


In [5]:
df_mmm.head(2)

,gene,logfoldchanges,pval,Wilcoxon_adj_pval
0,IGFBP7,1.407640,1.113706e-264,3.171388e-260
1,MT-ND4L,2.511231,1.445840e-165,1.372391e-161


In [6]:
# 1) 각 컬럼별 rank 계산 (값이 작을수록 1위, NaN은 가장 마지막)
df_mmm['rank_wilcoxon'] = df_mmm['Wilcoxon_adj_pval'] \
    .rank(method='min', ascending=True, na_option='bottom').astype(int)

# df_mmm['rank_limma'] = df_mmm['limmatrend_adj_pval'] \
#     .rank(method='min', ascending=True, na_option='bottom').astype(int)

# df_mmm['rank_RISCQP'] = df_mmm['RISCQP_adj_pval'] \
#     .rank(method='min', ascending=True, na_option='bottom').astype(int)

# df_mmm['rank_MAST'] = df_mmm['MAST_adj_pval'] \
#     .rank(method='min', ascending=True, na_option='bottom').astype(int)


# 2) 5개 rank의 합 계산
df_mmm['rank_sum'] = df_mmm[
    ['rank_wilcoxon', #'rank_limma', 'rank_RISCQP', 'rank_MAST'
     ]
].sum(axis=1)

# 3) rank_sum 기준으로 정렬
df_mmm = df_mmm.sort_values('rank_sum').reset_index(drop=True)

# 결과 확인 (선택사항)
df_mmm

,gene,logfoldchanges,pval,Wilcoxon_adj_pval,rank_wilcoxon,rank_sum
0,IGFBP7,1.407640,1.113706e-264,3.171388e-260,1,1
1,MT-RNR2,-4.585507,5.809130e-205,8.271040e-201,2,2
2,MT-ND4L,2.511231,1.445840e-165,1.372391e-161,3,3
3,MT-RNR1,-3.711353,7.409173e-159,5.274590e-155,4,4
4,ITM2B,0.923202,1.205370e-145,6.864824e-142,5,5
...,...,...,...,...,...,...
28471,VPS33A,1.190612,1.297864e-01,1.000000e+00,3645,3645
28472,POLE4,0.313383,1.297425e-01,1.000000e+00,3645,3645
28473,AIMP1,0.173227,1.294257e-01,1.000000e+00,3645,3645
28474,CEP85L,0.231634,1.289196e-01,1.000000e+00,3645,3645


In [7]:
# df_mmm_re = df_mmm[['rank_sum', 'gene', 'rank_wilcoxon', #'rank_limma', 'rank_RISCQP', 'rank_MAST'
#                     ]]
# df_mmm_re.head(20)

In [8]:
# 원하는 순서의 리스트
genes_of_interest = ["VSIG4", "LYVE1", "CLEC5A", "CX3CR1", "FN1", "IL11", "NRAP", "SDC1", "SCG5"]
df_mmm_filtered = df_mmm[df_mmm["gene"].isin(genes_of_interest)]
df_mmm_filtered

,gene,logfoldchanges,pval,Wilcoxon_adj_pval,rank_wilcoxon,rank_sum
516,FN1,-0.522099,1.512390e-09,8.330136e-08,517,517
7009,CLEC5A,0.452462,9.217283e-01,1.000000e+00,3645,3645
7494,VSIG4,0.821235,9.432427e-01,1.000000e+00,3645,3645
8200,SCG5,0.094739,9.614823e-01,1.000000e+00,3645,3645
9850,SDC1,0.142129,9.887818e-01,1.000000e+00,3645,3645
16367,NRAP,-16.788740,9.889253e-01,1.000000e+00,3645,3645
20003,CX3CR1,-19.514956,9.446689e-01,1.000000e+00,3645,3645
22102,LYVE1,-1.817548,8.576352e-01,1.000000e+00,3645,3645
22874,IL11,-1.304836,7.989232e-01,1.000000e+00,3645,3645


In [9]:
import pandas as pd
from functools import reduce

# 1️⃣ 각 메서드별 “gene + sign” DataFrame 생성
sign_dfs = []

# Wilcoxon: logfoldchanges >0 → MSI, <0 → MSS
sign_dfs.append(
    df_mmm_filtered[['gene','logfoldchanges']]
    .assign(Wilcoxon_sign=lambda d: d['logfoldchanges'].gt(0).map({True:'MSI', False:'MSS'}))
    [['gene','Wilcoxon_sign']]
)

# limmatrend: log2FC <0 → MSI, >0 → MSS
# sign_dfs.append(
#     df_limmatrend[['gene','logfoldchanges']]
#     .assign(limmat_sign=lambda d: d['logfoldchanges'].lt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','limmat_sign']]
# )

# RISCQP: log2FC <0 → MSI, >0 → MSS
# sign_dfs.append(
#     df_RISCQP[['gene','log2FC']]
#     .assign(RISCQP_sign=lambda d: d['log2FC'].lt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','RISCQP_sign']]
# )

# monocle: log2FC >0 → MSI, <0 → MSS
# sign_dfs.append(
#     df_monocle[['gene','log2FC']]
#     .assign(monocle_sign=lambda d: d['log2FC'].gt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','monocle_sign']]
# )

# # MAST: logfoldchanges >0 → MSI, <0 → MSS
# sign_dfs.append(
#     df_MAST[['gene','logfoldchanges']]
#     .assign(MAST_sign=lambda d: d['logfoldchanges'].gt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','MAST_sign']]
# )

# MAST: logfoldchanges >0 → MSI, <0 → MSS
# sign_dfs.append(
#     df_MAST[['gene','logfoldchanges']]
#     .assign(MAST_sign=lambda d: d['logfoldchanges'].lt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','MAST_sign']]
# )

# 2️⃣ 모든 sign DataFrame outer merge
sign_df = reduce(lambda a,b: pd.merge(a,b,on='gene',how='outer'), sign_dfs)

# 3️⃣ rank 합계 df와 결합
df_final = df_mmm_filtered.merge(sign_df, on='gene', how='left')

# 4️⃣ consensus: MSI_sign이 3개 이상 → MSI_high, else MSS_high
# sign_cols = [col for col in df_final.columns if col.endswith('_sign')]
# df_final['consensus'] = df_final[sign_cols].apply(
#     lambda row: 'MSI_high' if (row=='MSI').sum() >= 3 else 'MSS_high',
#     axis=1
# )

# 5️⃣ 결과 확인 (상위 20개)
print(df_final.sort_values('rank_sum').head(20))

     gene  logfoldchanges          pval  Wilcoxon_adj_pval  rank_wilcoxon  \
0     FN1       -0.522099  1.512390e-09       8.330136e-08            517   
1  CLEC5A        0.452462  9.217283e-01       1.000000e+00           3645   
2   VSIG4        0.821235  9.432427e-01       1.000000e+00           3645   
3    SCG5        0.094739  9.614823e-01       1.000000e+00           3645   
4    SDC1        0.142129  9.887818e-01       1.000000e+00           3645   
5    NRAP      -16.788740  9.889253e-01       1.000000e+00           3645   
6  CX3CR1      -19.514956  9.446689e-01       1.000000e+00           3645   
7   LYVE1       -1.817548  8.576352e-01       1.000000e+00           3645   
8    IL11       -1.304836  7.989232e-01       1.000000e+00           3645   

   rank_sum Wilcoxon_sign  
0       517           MSS  
1      3645           MSI  
2      3645           MSI  
3      3645           MSI  
4      3645           MSI  
5      3645           MSS  
6      3645           MSS  
7    

In [10]:
df_final

,gene,logfoldchanges,pval,Wilcoxon_adj_pval,rank_wilcoxon,rank_sum,Wilcoxon_sign
0,FN1,-0.522099,1.512390e-09,8.330136e-08,517,517,MSS
1,CLEC5A,0.452462,9.217283e-01,1.000000e+00,3645,3645,MSI
2,VSIG4,0.821235,9.432427e-01,1.000000e+00,3645,3645,MSI
3,SCG5,0.094739,9.614823e-01,1.000000e+00,3645,3645,MSI
4,SDC1,0.142129,9.887818e-01,1.000000e+00,3645,3645,MSI
5,NRAP,-16.788740,9.889253e-01,1.000000e+00,3645,3645,MSS
6,CX3CR1,-19.514956,9.446689e-01,1.000000e+00,3645,3645,MSS
7,LYVE1,-1.817548,8.576352e-01,1.000000e+00,3645,3645,MSS
8,IL11,-1.304836,7.989232e-01,1.000000e+00,3645,3645,MSS


In [12]:
import numpy as np

# 1) 기존 neg_log10_adj_pval 계산
epsilon = np.nextafter(0, 1)
df_final['neg_log10_adj_pval'] = -np.log10(
    df_final['Wilcoxon_adj_pval'].replace(0, epsilon)
)

# 2) 지수표현(예: 2.44e+02)으로 소수점 둘째 자리까지 포맷
df_final['neg_log10_adj_pval'] = df_final['neg_log10_adj_pval'] \
    .apply(lambda x: f"{x:.1e}")

# 3) gene 순서 재배열
desired_order = ["VSIG4", "LYVE1", "CLEC5A", "CX3CR1",
                 "FN1", "IL11", "NRAP", "SDC1", "SCG5"]
df_final = (
    df_final
    .set_index('gene')
    .reindex(desired_order)
    .reset_index()
)

# 확인
print(df_final[['gene', 'neg_log10_adj_pval', 'Wilcoxon_sign']])

     gene neg_log10_adj_pval Wilcoxon_sign
0   VSIG4           -0.0e+00           MSI
1   LYVE1           -0.0e+00           MSS
2  CLEC5A           -0.0e+00           MSI
3  CX3CR1           -0.0e+00           MSS
4     FN1            7.1e+00           MSS
5    IL11           -0.0e+00           MSS
6    NRAP           -0.0e+00           MSS
7    SDC1           -0.0e+00           MSI
8    SCG5           -0.0e+00           MSI
